# 03 - Tuning rapido de XGBoost

Replica el ajuste del profe en pequeA+-o. Permite guardar tus hiperparA metros en `artifacts/lpmc_xgb_custom_params.json` para usarlos en la libreta 4.

In [18]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data' / 'preprocessed'
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)
train = pd.read_csv(DATA_DIR / 'LPMC_train.csv')

target_col = 'travel_mode'
X = train.drop(columns=[target_col])
y = train[target_col].astype(int)
print('Dimensiones train:', X.shape)


Dimensiones train: (54766, 28)


In [19]:
best_prof_path = ARTIFACTS / 'lpmc_xgb_best_params.json'
with best_prof_path.open() as f:
    best_prof_bundle = json.load(f)
best_prof_params = best_prof_bundle['params']
scaled_features = best_prof_bundle['scaled_features']

print('Hiperparametros del profe (1000 evals, 5-CV):')
for k, v in best_prof_params.items():
    print(f"  {k}: {v}")


Hiperparametros del profe (1000 evals, 5-CV):
  max_depth: 4
  gamma: 3.931781434966718
  min_child_weight: 35
  max_delta_step: 9
  subsample: 0.7667332690572212
  colsample_bytree: 0.9343084611837867
  colsample_bylevel: 0.6514580147153639
  reg_alpha: 0.04829946881115557
  reg_lambda: 0.037112340554694714
  n_estimators: 4809
  learning_rate: 0.01
  objective: multi:softprob
  eval_metric: mlogloss
  num_class: 4
  n_jobs: -1
  random_state: 42


In [20]:
def gmpca_from_proba(proba, y_true):
    proba = np.asarray(proba)
    y_true = np.asarray(y_true)
    proba = np.clip(proba, 1e-12, 1.0)
    log_like = np.log(proba[np.arange(len(y_true)), y_true]).sum()
    return float(np.exp(log_like / len(y_true)))


In [21]:
RUN_TUNING = True  # activa para lanzar la busqueda

try:
    import hyperopt
    from hyperopt import hp, tpe, fmin, Trials
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from xgboost import XGBClassifier
    HAVE_HYPEROPT = True
except ImportError:
    HAVE_HYPEROPT = False
    print('hyperopt no esta instalado; usa `pip install hyperopt` si quieres buscar desde cero (lento).')

if RUN_TUNING and HAVE_HYPEROPT:
    space = {
        'max_depth': hp.quniform('max_depth', 3, 8, 1),
        'gamma': hp.loguniform('gamma', -3, 1),
        'min_child_weight': hp.quniform('min_child_weight', 1, 50, 1),
        'subsample': hp.uniform('subsample', 0.6, 0.9),
        'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 0.9),
        'reg_alpha': hp.loguniform('reg_alpha', -6, 1),
        'reg_lambda': hp.loguniform('reg_lambda', -6, 1),
        'n_estimators': hp.quniform('n_estimators', 200, 1200, 50),
        'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    }

    def objective(space):
        params = {k: int(v) if k in ['max_depth', 'min_child_weight', 'n_estimators'] else float(v)
                  for k, v in space.items()}
        params.update({
            'objective': 'multi:softprob',
            'eval_metric': 'mlogloss',
            'num_class': 4,
            'n_jobs': -1,
            'random_state': 42,
        })

        X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
        scaler = StandardScaler()
        X_tr_scaled = X_tr.copy()
        X_val_scaled = X_val.copy()
        X_tr_scaled[scaled_features] = scaler.fit_transform(X_tr_scaled[scaled_features])
        X_val_scaled[scaled_features] = scaler.transform(X_val_scaled[scaled_features])

        clf = XGBClassifier(**params)
        clf.fit(X_tr_scaled, y_tr)
        proba = clf.predict_proba(X_val_scaled)
        score = gmpca_from_proba(proba, y_val.to_numpy())
        return {'loss': -score, 'status': hyperopt.STATUS_OK}

    trials = Trials()
    best_search = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=10, trials=trials, rstate=np.random.default_rng(42))
    print('Resultado de la busqueda corta:', best_search)

    # Guardar tus hiperparametros en un JSON para la libreta 4
    best_custom = {k: int(v) if k in ['max_depth', 'min_child_weight', 'n_estimators'] else float(v)
                   for k, v in best_search.items()}
    best_custom.update({
        'objective': 'multi:softprob',
        'eval_metric': 'mlogloss',
        'num_class': 4,
        'n_jobs': -1,
        'random_state': 42,
    })

    out_path = ARTIFACTS / 'lpmc_xgb_custom_params.json'
    payload = {
        'source': '03_tune_xgb.ipynb',
        'params': best_custom,
        'scaled_features': scaled_features,
    }
    out_path.write_text(json.dumps(payload, indent=2))
    print('Hiperparametros propios guardados en:', out_path)
else:
    print('RUN_TUNING esta a False o falta hyperopt; no se ha buscado nada.')


100%|██████████| 10/10 [00:30<00:00,  3.04s/trial, best loss: -0.5960890054702759]
Resultado de la busqueda corta: {'colsample_bytree': np.float64(0.873849456868946), 'gamma': np.float64(0.1893686922910923), 'learning_rate': np.float64(0.2212640428030498), 'max_depth': np.float64(7.0), 'min_child_weight': np.float64(8.0), 'n_estimators': np.float64(550.0), 'reg_alpha': np.float64(0.4659161534440383), 'reg_lambda': np.float64(0.8274781529133294), 'subsample': np.float64(0.6192860589928647)}
Hiperparametros propios guardados en: f:\TFM\lpmc\artifacts\lpmc_xgb_custom_params.json
